# Whisper realtime server on a free Colab GPU

Runs this repo's WhisperLiveKit server on Colab's GPU and exposes it over an HTTPS
tunnel so your browser microphone can reach it.

**Set the GPU first:** menu **Runtime -> Change runtime type -> T4 GPU -> Save**.
Do this before running anything, because changing it restarts the session.

Then run every cell in order, top to bottom. The last cell prints a
`https://...trycloudflare.com` link. Open it, allow the mic, press record.

Keep this tab open - when the Colab session ends, the server dies with it.

## 1. Check the GPU is actually on

In [ ]:
!nvidia-smi --query-gpu=name,memory.total,driver_version --format=csv

# If this errors or prints nothing, the runtime has no GPU.
# Fix: Runtime -> Change runtime type -> T4 GPU -> Save, then run this cell again.

## 2. Install dependencies

Takes 2-3 minutes. Colab ships its own torch; pip may replace it. If Colab shows a
"restart session" button after this cell, click it, then **continue from cell 3**
(you do not need to re-run this one).

In [ ]:
!pip install -q faster-whisper whisperlivekit nvidia-cublas-cu12 nvidia-cudnn-cu12
print("installed")

## 3. Make the CUDA libraries visible

`ctranslate2` (the engine behind faster-whisper) loads cuBLAS and cuDNN at runtime.
The pip wheels place them where the loader does not look by default, so they have to
be added to `LD_LIBRARY_PATH` before the server process starts.

In [ ]:
import os, site

libs = []
for sp in site.getsitepackages():
    nv = os.path.join(sp, "nvidia")
    if os.path.isdir(nv):
        for sub in sorted(os.listdir(nv)):
            d = os.path.join(nv, sub, "lib")
            if os.path.isdir(d):
                libs.append(d)

os.environ["LD_LIBRARY_PATH"] = ":".join(libs + [os.environ.get("LD_LIBRARY_PATH", "")])
print(f"{len(libs)} CUDA lib dirs added")

import ctranslate2
n = ctranslate2.get_cuda_device_count()
print("CUDA devices visible to ctranslate2:", n)
if n == 0:
    print("!! 0 devices - the runtime has no GPU. Runtime -> Change runtime type -> T4 GPU.")

## 4. Get the code and apply the browser mic fix

`patch_webui.py` is **required**. Without it the page loads but no audio ever reaches
the server, and you get "No audio detected..." forever. See the repo README for the
details of what it fixes.

In [ ]:
import os, subprocess

REPO = "https://github.com/kunalost/whisper.git"
DIR = "/content/whisper"

if not os.path.isdir(DIR):
    subprocess.run(["git", "clone", "--depth", "1", REPO, DIR], check=True)
else:
    subprocess.run(["git", "-C", DIR, "pull"], check=True)

subprocess.run(["python", "patch_webui.py"], cwd=DIR, check=True)
print("repo ready at", DIR)

## 5. Install cloudflared (the tunnel)

Colab gives you no public inbound port, so your browser cannot reach `localhost:8000`.
cloudflared makes an outbound connection and hands back a public **https** URL.

HTTPS matters: browsers refuse microphone access on plain http, so an http tunnel
would load the page but never capture any audio.

In [ ]:
!wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -O /usr/local/bin/cloudflared
!chmod +x /usr/local/bin/cloudflared
!cloudflared --version

## 6. Start the server

First run downloads the model. `large-v3` is about 3 GB, so allow a few minutes.

Options you may want to change below:
- `--model large-v3-turbo` - noticeably faster, slightly less accurate
- `--language hi` - force Hindi. Auto-detect often picks the wrong language on short
  live audio, so set this if you are speaking Hindi.

In [ ]:
import subprocess, time, urllib.request, os

LOG = "/tmp/wlk.log"
args = [
    "python", "run_wlk.py",
    "--model", "large-v3",
    "--language", "auto",
    "--backend", "faster-whisper",
    "--backend-policy", "localagreement",
    "--port", "8000",
    "--pcm-input",
    "--min-chunk-size", "0.5",
]
env = dict(os.environ, WLK_DEVICE="cuda", WLK_COMPUTE="float16")

log = open(LOG, "w")
server = subprocess.Popen(args, cwd=DIR, env=env, stdout=log, stderr=subprocess.STDOUT)
print("server pid", server.pid, "- loading model, please wait...")

for i in range(90):
    if server.poll() is not None:
        print("SERVER EXITED. Log tail:")
        print(open(LOG).read()[-3000:])
        break
    try:
        with urllib.request.urlopen("http://localhost:8000/health", timeout=2) as r:
            print("READY:", r.read().decode())
            break
    except Exception:
        time.sleep(10)
else:
    print("timed out. Log tail:")
    print(open(LOG).read()[-3000:])

## 7. Open the tunnel

This cell runs forever on purpose - it is holding the tunnel open. Do not stop it
while you are using the app.

In the output look for a boxed line ending in `trycloudflare.com`. Open that
**https** URL in a new tab, allow the microphone, and press record.

In [ ]:
!cloudflared tunnel --url http://localhost:8000 --no-autoupdate

## Troubleshooting

| Symptom | Cause / fix |
|---|---|
| `"No audio detected..."` on the page | Cell 4 did not run. Re-run it, then hard-refresh the tunnel page with Ctrl+Shift+R. |
| Browser never asks for mic permission | You opened the `http://` link. Use the `https://` one. |
| `CUDA devices visible: 0` in cell 3 | No GPU on the runtime. Runtime -> Change runtime type -> T4 GPU, then re-run from cell 1. |
| Cell 6 prints `SERVER EXITED` | Read the log tail it prints. A cuDNN/cuBLAS `cannot open shared object file` error means cell 3 was skipped. |
| `Cannot connect to GPU backend` | Colab has no free GPU right now, or you hit your quota. Wait and retry, or use Kaggle. |
| Everything stops suddenly | Session hit its time or idle limit. Normal on the free tier - re-run the notebook. |

**Alternative to cloudflared:** Colab can proxy a port itself:

```python
from google.colab.output import eval_js
print(eval_js("google.colab.kernel.proxyPort(8000)"))
```

That avoids a third-party tunnel, but I have not confirmed it forwards WebSocket
upgrades, which this app needs. Try it if you like; use cloudflared if it fails.